In [1]:
import pandas as pd

INPUT_FILE = "properties_karachi_dataset.xlsx"
OUTPUT_FILE = "properties_karachi_dataset_sorted.xlsx"
DATA_SHEET = "Property Ownership Data"
TYPE_ORDER = ["Apartment", "Portion", "Residential", "House", "Plot"]

sheets = pd.read_excel(INPUT_FILE, sheet_name=None)
df = sheets[DATA_SHEET]
for col in ["Area", "Street Number", "Property Type"]:
    df[col] = df[col].str.strip()

df["Property Type"] = pd.Categorical(df["Property Type"], categories=TYPE_ORDER, ordered=True)
df["_street_no"] = df["Street Number"].str.extract(r"(\d+)", expand=False).astype(int)

df = df.sort_values(
    by=["Area", "Property Type", "Block", "_street_no", "Owner ID"],
    ascending=True,
    kind="stable",
).drop(columns="_street_no")

df["Property Type"] = df["Property Type"].astype(str)
df = df.reset_index(drop=True)
sheets[DATA_SHEET] = df

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    for name, sheet_df in sheets.items():
        sheet_df.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        ws.freeze_panes = "A2"
        for i, col in enumerate(sheet_df.columns, start=1):
            width = max(sheet_df[col].astype(str).str.len().max(), len(str(col))) + 2
            ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = min(width, 60)

print(f"Saved {len(df)} rows to {OUTPUT_FILE}")

Saved 16000 rows to properties_karachi_dataset_sorted.xlsx


In [1]:
import pandas as pd

INPUT_FILE = "properties_karachi_dataset_sorted.xlsx"
DATA_SHEET = "Property Ownership Data"
TYPES = ["Apartment", "Portion", "Commercial", "House", "Plot"]

count= 0
df = pd.read_excel(INPUT_FILE, sheet_name=DATA_SHEET)
df["Property Type"] = df["Property Type"].str.strip()

for prop_type in TYPES:
    part = df[df["Property Type"] == prop_type]
    out_file = f"{prop_type.lower()}s.csv"
    part.to_csv(out_file, index=False, encoding="utf-8-sig")
    print(f"{out_file}: {len(part)} rows")
    count = count + len(part)
print(f"Total count: {count} rows")

apartments.csv: 3590 rows
portions.csv: 2519 rows
commercials.csv: 2991 rows
houses.csv: 3585 rows
plots.csv: 3315 rows
Total count: 16000 rows
